# ROC Threshold Breakout on SPY
## Strategy Brief
The ROC (Rate of Change) Threshold Breakout strategy is a momentum-based trading approach applied to SPY, the ETF tracking the S&P 500. The strategy generates buy signals when the ROC exceeds a certain threshold, indicating strong upward momentum. Conversely, sell signals are triggered when the ROC falls below a negative threshold, suggesting downward momentum. This strategy aims to capitalize on significant price movements and outperform a buy-and-hold approach by capturing breakout trends.
## References
- https://www.rocskincare.com/

In [ ]:
!pip install yfinance pandas numpy matplotlib scipy

## PHASE 1 - Trading Context
In this phase, we define the parameters for our trading strategy. These parameters include the ROC period, the positive and negative thresholds for generating signals, and the initial capital for backtesting.

In [ ]:
ROC_PERIOD = 14
POSITIVE_THRESHOLD = 5
NEGATIVE_THRESHOLD = -5
INITIAL_CAPITAL = 10000

## PHASE 2 - Data Exploration
We will download historical price data for SPY from Yahoo Finance, calculate the ROC indicator, and plot it alongside the price data to visualize potential breakout points.

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Download SPY data
data = yf.download('SPY', start='2010-01-01')

# Calculate ROC
roc = data['Adj Close'].pct_change(ROC_PERIOD) * 100

# Plot
plt.figure(figsize=(14, 7))
plt.subplot(2, 1, 1)
plt.plot(data['Adj Close'], label='SPY Price')
plt.title('SPY Price and ROC')
plt.legend()
plt.subplot(2, 1, 2)
plt.plot(roc, label='ROC', color='orange')
plt.axhline(POSITIVE_THRESHOLD, color='green', linestyle='--', label='Positive Threshold')
plt.axhline(NEGATIVE_THRESHOLD, color='red', linestyle='--', label='Negative Threshold')
plt.legend()
plt.show()

## PHASE 3 - Strategy Engineering
We will create a signal series based on the ROC indicator. Buy signals occur when the ROC crosses above the positive threshold, and sell signals occur when the ROC crosses below the negative threshold. We will use these signals to determine our positions.

In [ ]:
signals = pd.Series(index=roc.index, data=0)
signals[roc > POSITIVE_THRESHOLD] = 1
signals[roc < NEGATIVE_THRESHOLD] = -1

# Generate positions
positions = signals.replace(to_replace=0, method='ffill').fillna(0)

## PHASE 4 - Coding & Backtesting
We will backtest the strategy by calculating daily returns based on the positions and plotting the resulting equity curve.

In [ ]:
daily_returns = data['Adj Close'].pct_change()
strategy_returns = daily_returns * positions.shift(1)
equity_curve = (1 + strategy_returns).cumprod() * INITIAL_CAPITAL

# Plot equity curve
plt.figure(figsize=(14, 7))
plt.plot(equity_curve, label='Strategy Equity Curve')
plt.title('Equity Curve')
plt.legend()
plt.show()

## PHASE 5 - Performance Evaluation
We will evaluate the performance of the strategy using metrics such as CAGR, Sharpe ratio, Sortino ratio, Calmar ratio, and maximum drawdown. A comparison will be made against a buy-and-hold strategy.

In [ ]:
def calculate_performance_metrics(equity_curve):
    cagr = (equity_curve[-1] / equity_curve[0]) ** (252 / len(equity_curve)) - 1
    daily_returns = equity_curve.pct_change().dropna()
    sharpe_ratio = np.sqrt(252) * daily_returns.mean() / daily_returns.std()
    downside_std = daily_returns[daily_returns < 0].std()
    sortino_ratio = np.sqrt(252) * daily_returns.mean() / downside_std
    max_drawdown = (equity_curve / equity_curve.cummax() - 1).min()
    calmar_ratio = cagr / abs(max_drawdown)
    return cagr, sharpe_ratio, sortino_ratio, calmar_ratio, max_drawdown

strategy_metrics = calculate_performance_metrics(equity_curve)
buy_and_hold_metrics = calculate_performance_metrics((1 + daily_returns).cumprod() * INITIAL_CAPITAL)

comparison_table = pd.DataFrame({
    'Metric': ['CAGR', 'Sharpe Ratio', 'Sortino Ratio', 'Calmar Ratio', 'Max Drawdown'],
    'Strategy': strategy_metrics,
    'Buy and Hold': buy_and_hold_metrics
})
print(comparison_table)

## PHASE 6 - Deploy & Monitor
We will create a function to download the last 60 days of SPY data, compute today's ROC signal, and print the recommended position.

In [ ]:
def get_latest_signal():
    latest_data = yf.download('SPY', period='60d')
    latest_roc = latest_data['Adj Close'].pct_change(ROC_PERIOD) * 100
    latest_signal = 0
    if latest_roc.iloc[-1] > POSITIVE_THRESHOLD:
        latest_signal = 1
    elif latest_roc.iloc[-1] < NEGATIVE_THRESHOLD:
        latest_signal = -1
    print(f"Today's recommended position: {'Buy' if latest_signal == 1 else 'Sell' if latest_signal == -1 else 'Hold'}")

get_latest_signal()